In [2]:
!pip install -q transformers datasets evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score,f1_score

In [4]:
dataset = load_dataset("NeuronZero/Symptom2Disease")

dataset

README.md:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

Symptom2Disease.csv:   0%|          | 0.00/230k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1200 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'label', 'text'],
        num_rows: 1200
    })
})

In [5]:
df = dataset["train"].to_pandas()

df.sample(5)

,Unnamed: 0,label,text
1037,137,gastroesophageal reflux disease,I regularly wake up with a scratchy voice or a...
817,217,Jaundice,"The itch, the nausea, and the weariness have b..."
608,8,Bronchial Asthma,"Mom, Dad, I've been feeling really tired and w..."
128,128,Typhoid,I've been suffering from constipation and stom...
522,222,Arthritis,My neck has been extremely tight and my muscle...


In [6]:
df = df.drop(columns=["Unnamed: 0"])

In [7]:
df.info()

df.head()

df["label"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   1200 non-null   object
 1   text    1200 non-null   object
dtypes: object(2)
memory usage: 18.9+ KB


,count
label,
Psoriasis,50
Varicose Veins,50
Typhoid,50
Chicken pox,50
Impetigo,50
Dengue,50
Fungal infection,50
Common Cold,50
Pneumonia,50


In [8]:
df.isnull().sum()

,0
label,0
text,0


In [9]:
print(df.duplicated().sum())

47


In [10]:
df = df.drop_duplicates()

In [11]:
encoder = LabelEncoder()

df["label_id"] = encoder.fit_transform(df["label"])

In [12]:
from sklearn.model_selection import train_test_split

train_df,test_df = train_test_split(

    df,

    test_size=0.2,

    random_state=42,

    stratify=df["label_id"]

)

In [13]:
MODEL_NAME="emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(

    MODEL_NAME,

    num_labels=len(encoder.classes_)

)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the chec

In [14]:
MAX_LENGTH = 128

def tokenize(batch):

    return tokenizer(

        batch["text"],

        truncation=True,

        padding="max_length",

        max_length=MAX_LENGTH

    )

In [15]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)

test_dataset = Dataset.from_pandas(test_df)

train_dataset = train_dataset.map(tokenize, batched=True)

test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/922 [00:00<?, ? examples/s]

Map:   0%|          | 0/231 [00:00<?, ? examples/s]

In [16]:
train_dataset = train_dataset.rename_column(
    "label_id",
    "labels"
)

test_dataset = test_dataset.rename_column(
    "label_id",
    "labels"
)

In [17]:
train_dataset.set_format(

    type="torch",

    columns=[

        "input_ids",

        "attention_mask",

        "labels"

    ]

)

test_dataset.set_format(

    type="torch",

    columns=[

        "input_ids",

        "attention_mask",

        "labels"

    ]

)

In [18]:
import evaluate

accuracy = evaluate.load("accuracy")

f1 = evaluate.load("f1")

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

In [19]:
import numpy as np

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(

        logits,

        axis=-1

    )

    acc = accuracy.compute(

        predictions=predictions,

        references=labels

    )

    f1_score = f1.compute(

        predictions=predictions,

        references=labels,

        average="weighted"

    )

    return {

        "accuracy": acc["accuracy"],

        "f1": f1_score["f1"]

    }

In [20]:
training_args = TrainingArguments(

    output_dir="./clinicalbert_model",

    save_strategy="best",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=5,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="accuracy",

    logging_steps=20,

    report_to="none"

)

In [21]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics

)

In [23]:
#trainer.train()

In [24]:
train_df.columns

Index(['label', 'text', 'label_id'], dtype='object')

In [26]:
train_df.head()

,label,text,label_id
919,urinary tract infection,"My stomach hurts, and I frequently acquire a t...",23
687,Hypertension,I was at work when I started feeling a headach...,9
43,Psoriasis,My fingers and soles have pretty thick skin th...,15
486,Dimorphic Hemorrhoids,My bowel motions are giving me a lot of proble...,7
379,Common Cold,"For days, I've had a nasty cough and cold. My ...",5


In [27]:
print(type(model))

<class 'transformers.models.bert.modeling_bert.BertForSequenceClassification'>


In [28]:
print(model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [29]:
from torch.utils.data import Dataset

class SymptomDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_len=128):

        self.texts = dataframe["text"].values
        self.labels = dataframe["label_id"].values
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoding = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [30]:
train_dataset = SymptomDataset(train_df, tokenizer)

test_dataset = SymptomDataset(test_df, tokenizer)

In [31]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

In [32]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

model.to(device)

cuda


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [33]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

In [34]:
from tqdm import tqdm

EPOCHS = 5

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    progress = tqdm(train_loader)

    for batch in progress:

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        progress.set_description(
            f"Epoch {epoch+1}"
        )

        progress.set_postfix(
            loss=loss.item()
        )

    avg_loss = total_loss / len(train_loader)

    print(f"\nEpoch {epoch+1} Average Loss: {avg_loss:.4f}")

Epoch 1: 100%|██████████| 58/58 [00:20<00:00,  2.84it/s, loss=2.85]



Epoch 1 Average Loss: 3.0968


Epoch 2: 100%|██████████| 58/58 [00:20<00:00,  2.80it/s, loss=1.92]



Epoch 2 Average Loss: 2.4404


Epoch 3: 100%|██████████| 58/58 [00:21<00:00,  2.66it/s, loss=1.29]



Epoch 3 Average Loss: 1.6148


Epoch 4: 100%|██████████| 58/58 [00:23<00:00,  2.51it/s, loss=0.679]



Epoch 4 Average Loss: 0.9891


Epoch 5: 100%|██████████| 58/58 [00:24<00:00,  2.41it/s, loss=0.424]


Epoch 5 Average Loss: 0.5732


In [35]:
model.save_pretrained("Disease_Model")

tokenizer.save_pretrained("Disease_Model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('Disease_Model/tokenizer_config.json', 'Disease_Model/tokenizer.json')

In [36]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate(model, dataloader):

    model.eval()

    predictions = []
    actual_labels = []

    total_loss = 0

    with torch.no_grad():

        for batch in dataloader:

            input_ids = batch["input_ids"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss

            total_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=1)

            predictions.extend(preds.cpu().numpy())

            actual_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(actual_labels, predictions)

    f1 = f1_score(
        actual_labels,
        predictions,
        average="weighted"
    )

    avg_loss = total_loss / len(dataloader)

    return avg_loss, accuracy, f1

In [37]:
best_accuracy = 0.0
best_f1 = 0.0

patience = 2
counter = 0

In [38]:
from tqdm import tqdm

EPOCHS = 5

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    progress = tqdm(train_loader)

    for batch in progress:

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        labels = batch["labels"].to(device)

        outputs = model(

            input_ids=input_ids,

            attention_mask=attention_mask,

            labels=labels

        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        progress.set_postfix(
            loss=loss.item()
        )

    train_loss = total_loss / len(train_loader)

    val_loss, accuracy, f1 = evaluate(
        model,
        test_loader
    )

    print("="*50)

    print(f"Epoch {epoch+1}")

    print(f"Train Loss : {train_loss:.4f}")

    print(f"Validation Loss : {val_loss:.4f}")

    print(f"Accuracy : {accuracy:.4f}")

    print(f"F1 Score : {f1:.4f}")

    if accuracy > best_accuracy:

        best_accuracy = accuracy

        model.save_pretrained(
            "Best_Disease_Model"
        )

        tokenizer.save_pretrained(
            "Best_Disease_Model"
        )

        print("✅ Best Model Saved")

100%|██████████| 58/58 [00:24<00:00,  2.40it/s, loss=0.24]


Epoch 1
Train Loss : 0.3389
Validation Loss : 0.3389
Accuracy : 0.9740
F1 Score : 0.9736


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Best Model Saved


100%|██████████| 58/58 [00:23<00:00,  2.42it/s, loss=0.224]


...

Epoch 4
Train Loss : 0.1033
Validation Loss : 0.1749
Accuracy : 0.9827
F1 Score : 0.9823


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Best Model Saved


100%|██████████| 58/58 [00:23<00:00,  2.48it/s, loss=0.0714]


Epoch 5
Train Loss : 0.0815
Validation Loss : 0.1601
Accuracy : 0.9784
F1 Score : 0.9774


In [39]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [40]:
import os

save_path = "/content/drive/MyDrive/CareMate_AI"

os.makedirs(save_path, exist_ok=True)

print("Folder Created:", save_path)

Folder Created: /content/drive/MyDrive/CareMate_AI


In [41]:
import shutil

shutil.copytree(
    "Best_Disease_Model",
    f"{save_path}/Best_Disease_Model",
    dirs_exist_ok=True
)

print("✅ Model Saved to Google Drive")

✅ Model Saved to Google Drive


In [42]:
shutil.copytree(
    "Disease_Model",
    f"{save_path}/Disease_Model",
    dirs_exist_ok=True
)

'/content/drive/MyDrive/CareMate_AI/Disease_Model'

In [43]:
train_df.to_csv(f"{save_path}/train_df.csv", index=False)

test_df.to_csv(f"{save_path}/test_df.csv", index=False)

In [45]:
import joblib

joblib.dump(
    encoder,
    f"{save_path}/label_encoder.pkl"
)

['/content/drive/MyDrive/CareMate_AI/label_encoder.pkl']

In [46]:
import os

for file in os.listdir():
    print(file)

.config
Best_Disease_Model
drive
clinicalbert_model
Disease_Model
sample_data


In [47]:
import shutil

shutil.copy(
    "filename.ext",
    f"{save_path}/filename.ext"
)

FileNotFoundError: [Errno 2] No such file or directory: 'filename.ext'